In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from pyprojroot import here
from scipy import stats


sns.set_theme(style='whitegrid', font_scale=1.1)

ROOT = here()
DATA_DIR = ROOT / 'data'
OUTPUT_DIR = ROOT / 'outputs'
OUTPUT_DIR.mkdir(exist_ok=True)

In [ ]:
# ---- File paths (adjust if needed) ----
imd_2010_path       = DATA_DIR / 'imd_2010.xls'
imd_2019_path       = DATA_DIR / 'imd_2019.csv'
census_od_2021_path = DATA_DIR / 'ODMG01EW_MSOA.csv'
lookup_path         = DATA_DIR / 'NSPCL_NOV22_UK_LU.csv'
msoa_lookup_path    = DATA_DIR / 'msoa_2011_to_2021_lookup.csv'

In [ ]:
# ---- Load IMD ----
imd_2019 = pd.read_csv(imd_2019_path)
imd_2019.columns = imd_2019.columns.str.strip()

imd_2010 = pd.read_excel(imd_2010_path, sheet_name='IMD 2010')
imd_2010.columns = imd_2010.columns.str.strip()

# ---- Load Census O-D ----
census_od_2021 = pd.read_csv(census_od_2021_path)

# ---- Load Lookups ----
lookup = pd.read_csv(lookup_path, encoding='ISO-8859-1', low_memory=False)
msoa_11_21 = pd.read_csv(msoa_lookup_path)

# Quick overview
for name, df in [('IMD 2010', imd_2010), ('IMD 2019', imd_2019),
                  ('Census O-D 2021', census_od_2021), ('MSOA Lookup', msoa_11_21)]:
    print(f'\n===== {name} =====')
    print(f'Shape: {df.shape}')
    print(f'Columns: {df.columns.tolist()}')
    display(df.head(2))

In [ ]:
msoa_11_21.columns = msoa_11_21.columns.str.strip().str.lower()
print('MSOA lookup columns:', msoa_11_21.columns.tolist())
print('\nChange indicator counts:')
print(msoa_11_21['chngind'].value_counts())

In [ ]:
# Replace the chgind filter with:
unchanged = msoa_11_21[msoa_11_21['msoa11cd'] == msoa_11_21['msoa21cd']].copy()
msoa21_to_11 = dict(zip(unchanged['msoa21cd'], unchanged['msoa11cd']))

print(f'Total MSOAs in lookup: {len(msoa_11_21)}')
print(f'Unchanged (usable):    {len(unchanged)}')
print(f'Dropped (split/merged): {len(msoa_11_21) - len(unchanged)}')

In [ ]:
# Convert 2021 O-D data to 2011 MSOA codes
ORIGIN_COL = 'Migrant MSOA one year ago code'
DEST_COL   = 'Middle layer Super Output Areas code'

census_od_2021['origin_msoa11'] = census_od_2021[ORIGIN_COL].map(msoa21_to_11)
census_od_2021['dest_msoa11']   = census_od_2021[DEST_COL].map(msoa21_to_11)

n_total = len(census_od_2021)
n_mapped = census_od_2021[['origin_msoa11', 'dest_msoa11']].notna().all(axis=1).sum()
print(f'2021 O-D records: {n_total:,}')
print(f'Both endpoints mapped to 2011 codes: {n_mapped:,} ({n_mapped/n_total*100:.1f}%)')

In [ ]:
london_boroughs = [
    'City of London', 'Barking and Dagenham', 'Barnet', 'Bexley', 'Brent',
    'Bromley', 'Camden', 'Croydon', 'Ealing', 'Enfield', 'Greenwich',
    'Hackney', 'Hammersmith and Fulham', 'Haringey', 'Harrow', 'Havering',
    'Hillingdon', 'Hounslow', 'Islington', 'Kensington and Chelsea',
    'Kingston upon Thames', 'Lambeth', 'Lewisham', 'Merton', 'Newham',
    'Redbridge', 'Richmond upon Thames', 'Southwark', 'Sutton',
    'Tower Hamlets', 'Waltham Forest', 'Wandsworth', 'Westminster'
]

london_lookup = (
    lookup[lookup['ladnm'].isin(london_boroughs)]
    [['lsoa11cd', 'msoa11cd', 'ladnm']]
    .drop_duplicates()
)

print(f'London LSOAs: {london_lookup["lsoa11cd"].nunique()}')
print(f'London MSOAs: {london_lookup["msoa11cd"].nunique()}')
print(f'Boroughs:     {london_lookup["ladnm"].nunique()}')

In [ ]:
# ---- IMD 2010 → MSOA ----
IMD_2010_LSOA_COL  = 'LSOA CODE'   
IMD_2010_SCORE_COL = 'IMD SCORE'    

imd_2010_london = pd.merge(
    imd_2010[[IMD_2010_LSOA_COL, IMD_2010_SCORE_COL]],
    london_lookup,
    left_on=IMD_2010_LSOA_COL, right_on='lsoa11cd'
)
msoa_imd_2010 = (
    imd_2010_london
    .groupby(['msoa11cd', 'ladnm'])[IMD_2010_SCORE_COL]
    .mean().reset_index()
    .rename(columns={IMD_2010_SCORE_COL: 'IMD_2010'})
)

# ---- IMD 2019 → MSOA ----
imd_2019_london = pd.merge(
    imd_2019[['LSOA code (2011)', 'Index of Multiple Deprivation (IMD) Score']],
    london_lookup,
    left_on='LSOA code (2011)', right_on='lsoa11cd'
)
msoa_imd_2019 = (
    imd_2019_london
    .groupby('msoa11cd')['Index of Multiple Deprivation (IMD) Score']
    .mean().reset_index()
    .rename(columns={'Index of Multiple Deprivation (IMD) Score': 'IMD_2019'})
)

# ---- Combine ----
msoa_wealth = pd.merge(msoa_imd_2010, msoa_imd_2019, on='msoa11cd', how='inner')
msoa_wealth['IMD_Change'] = msoa_wealth['IMD_2019'] - msoa_wealth['IMD_2010']

print(f'London MSOAs with both IMD years: {len(msoa_wealth)}')
msoa_wealth.head()

In [ ]:
msoa_wealth['Wealth_Decile'] = pd.qcut(
    msoa_wealth['IMD_2010'], 10, labels=False
) + 1
msoa_wealth['Wealth_Decile'] = 11 - msoa_wealth['Wealth_Decile']
# 1 = most deprived, 10 = least deprived (wealthiest)

wealth_dict = msoa_wealth.set_index('msoa11cd')['Wealth_Decile'].to_dict()

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].hist(msoa_wealth['IMD_2010'], bins=30, color='steelblue', edgecolor='white')
axes[0].set_xlabel('Mean IMD 2010 Score (higher = more deprived)')
axes[0].set_ylabel('Number of MSOAs')
axes[0].set_title('Chart 1A: IMD 2010 Score Distribution (London MSOAs)')

decile_counts = msoa_wealth['Wealth_Decile'].value_counts().sort_index()

avg_msoa_count = decile_counts.mean()
print(f"Average number of MSOAs per decile: {avg_msoa_count}")

axes[1].bar(decile_counts.index, decile_counts.values, color='teal', edgecolor='white')
axes[1].set_xlabel('Wealth Decile (1 = most deprived, 10 = wealthiest)')
axes[1].set_ylabel('Number of MSOAs')
axes[1].set_title('Chart 1B: MSOA Count per Wealth Decile (2010 Baseline)')
axes[1].set_xticks(range(1, 11))

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'fig1_imd_baseline.png', dpi=150, bbox_inches='tight')
plt.show()